## 1. Imports

In [13]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import sys
import os
from pathlib import Path

# Ensure parent directory (src_GAMMA) is on sys.path for imports
sys.path.append(os.path.abspath(".."))

# IntermediateFusion models (corrected package path)
from IntermediateFusion.IntermediateFusionMLP import *
from IntermediateFusion.IntermediateFusionCNN import *
from IntermediateFusion.utils import cross_validation_5fold_intermediate_fusion

# Embedding extractors
from EmbeddingExtractors.ImageEmbeddingExtractor import ImageEmbeddingExtractor
from EmbeddingExtractors.TimeSeriesEmbeddingExtractor import TimeSeriesEmbeddingExtractor

## 2. Data Loading

In [ ]:
# [TODO] Define paths to CSV files
patient_data_csv = "path/to/patient_data.csv"
labels_csv = "path/to/labels.csv"
train_folds_csv = "path/to/train_folds.csv"
val_test_folds_csv = "path/to/val_test_folds.csv"

# Load data
patient_data = pd.read_csv(patient_data_csv)
labels = pd.read_csv(labels_csv)
train_folds = pd.read_csv(train_folds_csv)
val_test_folds = pd.read_csv(val_test_folds_csv)

print(f"Patient data shape: {patient_data.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Train folds shape: {train_folds.shape}")
print(f"Val/Test folds shape: {val_test_folds.shape}")

## 3. Embedding Extraction

In [ ]:
# [TODO] Initialize embedding extractors
image_extractor = ImageEmbeddingExtractor(
    model_name="resnet50",  # or other model
    pretrained=True
)

timeseries_extractor = TimeSeriesEmbeddingExtractor(
    model_name="default",  # or other model
    pretrained=True
)

# [TODO] Extract embeddings for image data
print("Extracting image embeddings...")
image_embeddings = image_extractor.extract(
    patient_data  # Adjust based on actual data structure
)
print(f"Image embeddings shape: {image_embeddings.shape}")

# [TODO] Extract embeddings for time series data
print("\nExtracting time series embeddings...")
timeseries_embeddings = timeseries_extractor.extract(
    patient_data  # Adjust based on actual data structure
)
print(f"Time series embeddings shape: {timeseries_embeddings.shape}")

## 4. Concatenation

In [ ]:
# Concatenate embeddings
concatenated_features = np.concatenate(
    [image_embeddings, timeseries_embeddings],
    axis=1
)
print(f"Concatenated features shape: {concatenated_features.shape}")

# [TODO] Save concatenated features to CSV if needed
features_df = pd.DataFrame(concatenated_features)
features_csv = "path/to/concatenated_features.csv"
# features_df.to_csv(features_csv, index=False)

## 5. Cross-Validation

In [ ]:
# [TODO] Define output directory for models
output_dir = "path/to/models"

# Run 5-fold cross-validation with MLP
print("Running 5-fold CV with IntermediateFusionMLP...")
best_model_path_mlp, metrics_history_mlp = cross_validation_5fold_intermediate_fusion(
    features_csv=features_csv,
    labels_csv=labels_csv,
    train_folds_csv=train_folds_csv,
    val_test_folds_csv=val_test_folds_csv,
    output_dir=output_dir,
    NN_type="MLP",
    hidden_dims=[512, 256, 128],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    verbose=True
)
print(f"Best MLP model saved at: {best_model_path_mlp}")

In [ ]:
# Run 5-fold cross-validation with CNN
print("Running 5-fold CV with IntermediateFusionCNN...")
best_model_path_cnn, metrics_history_cnn = cross_validation_5fold_intermediate_fusion(
    features_csv=features_csv,
    labels_csv=labels_csv,
    train_folds_csv=train_folds_csv,
    val_test_folds_csv=val_test_folds_csv,
    output_dir=output_dir,
    NN_type="CNN",
    hidden_dims=[512, 256],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    pooling_type="max",
    verbose=True
)
print(f"Best CNN model saved at: {best_model_path_cnn}")

## 6. Results Visualization

In [ ]:
# Set up matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Prepare data for visualization
mlp_folds = metrics_history_mlp['folds']
mlp_acc = metrics_history_mlp['test_acc']
mlp_f1 = metrics_history_mlp['test_f1']
mlp_auc = metrics_history_mlp['test_roc_auc']
mlp_loss = metrics_history_mlp['test_loss']

cnn_folds = metrics_history_cnn['folds']
cnn_acc = metrics_history_cnn['test_acc']
cnn_f1 = metrics_history_cnn['test_f1']
cnn_auc = metrics_history_cnn['test_roc_auc']
cnn_loss = metrics_history_cnn['test_loss']

print("MLP Results:")
print(f"  Accuracy: {np.mean(mlp_acc):.4f} ± {np.std(mlp_acc):.4f}")
print(f"  F1 Score: {np.mean(mlp_f1):.4f} ± {np.std(mlp_f1):.4f}")
print(f"  ROC AUC:  {np.mean(mlp_auc):.4f} ± {np.std(mlp_auc):.4f}")
print(f"  Loss:     {np.mean(mlp_loss):.4f} ± {np.std(mlp_loss):.4f}")

print("\nCNN Results:")
print(f"  Accuracy: {np.mean(cnn_acc):.4f} ± {np.std(cnn_acc):.4f}")
print(f"  F1 Score: {np.mean(cnn_f1):.4f} ± {np.std(cnn_f1):.4f}")
print(f"  ROC AUC:  {np.mean(cnn_auc):.4f} ± {np.std(cnn_auc):.4f}")
print(f"  Loss:     {np.mean(cnn_loss):.4f} ± {np.std(cnn_loss):.4f}")

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-Validation Results Comparison: MLP vs CNN', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
axes[0, 0].plot(mlp_folds, mlp_acc, marker='o', label='MLP', linewidth=2, markersize=8)
axes[0, 0].plot(cnn_folds, cnn_acc, marker='s', label='CNN', linewidth=2, markersize=8)
axes[0, 0].axhline(np.mean(mlp_acc), color='C0', linestyle='--', alpha=0.5)
axes[0, 0].axhline(np.mean(cnn_acc), color='C1', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Test Accuracy per Fold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(mlp_folds, mlp_f1, marker='o', label='MLP', linewidth=2, markersize=8)
axes[0, 1].plot(cnn_folds, cnn_f1, marker='s', label='CNN', linewidth=2, markersize=8)
axes[0, 1].axhline(np.mean(mlp_f1), color='C0', linestyle='--', alpha=0.5)
axes[0, 1].axhline(np.mean(cnn_f1), color='C1', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Test F1 Score per Fold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: ROC AUC
axes[1, 0].plot(mlp_folds, mlp_auc, marker='o', label='MLP', linewidth=2, markersize=8)
axes[1, 0].plot(cnn_folds, cnn_auc, marker='s', label='CNN', linewidth=2, markersize=8)
axes[1, 0].axhline(np.mean(mlp_auc), color='C0', linestyle='--', alpha=0.5)
axes[1, 0].axhline(np.mean(cnn_auc), color='C1', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('ROC AUC')
axes[1, 0].set_title('Test ROC AUC per Fold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loss
axes[1, 1].plot(mlp_folds, mlp_loss, marker='o', label='MLP', linewidth=2, markersize=8)
axes[1, 1].plot(cnn_folds, cnn_loss, marker='s', label='CNN', linewidth=2, markersize=8)
axes[1, 1].axhline(np.mean(mlp_loss), color='C0', linestyle='--', alpha=0.5)
axes[1, 1].axhline(np.mean(cnn_loss), color='C1', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Test Loss per Fold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create summary statistics plot
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'F1 Score', 'ROC AUC', 'Loss']
mlp_means = [np.mean(mlp_acc), np.mean(mlp_f1), np.mean(mlp_auc), np.mean(mlp_loss)]
mlp_stds = [np.std(mlp_acc), np.std(mlp_f1), np.std(mlp_auc), np.std(mlp_loss)]
cnn_means = [np.mean(cnn_acc), np.mean(cnn_f1), np.mean(cnn_auc), np.mean(cnn_loss)]
cnn_stds = [np.std(cnn_acc), np.std(cnn_f1), np.std(cnn_auc), np.std(cnn_loss)]

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, mlp_means, width, label='MLP', yerr=mlp_stds, capsize=5, alpha=0.8)
ax.bar(x + width/2, cnn_means, width, label='CNN', yerr=cnn_stds, capsize=5, alpha=0.8)

ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Average Cross-Validation Performance: MLP vs CNN', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()